# 🚀 01: Simulate

Population-grounded user simulation. Reads a panel of personas (or samples them inline per locale), runs the conversation simulator plugin, and writes a directory-partitioned trajectory dataset that downstream notebooks (`02_evaluate_simulation`) and the reporting layer consume.

Workflow: prototype locally with a small row count, then inspect the sanity checks and conversation previews before scaling the row count up.

This notebook is intentionally thin: it imports the same pipeline factories the `usersim` CLI does so the two flows stay byte-identical. For unattended runs use the CLI:

```bash
usersim simulate --locale en_US --num-rows 20 --out output/trajectories
```

For interactive iteration the notebook is more convenient: small `N` previews, locale-by-locale debugging, and live cell editing without re-launching a Python process.

**What you need:**

- `NVIDIA_API_KEY` exported in your shell (or set in the cell below).
- The Nemotron-Personas dataset for the locales you care about. The setup cell below downloads any missing locales the first time via the NGC CLI; if `ngc` isn't on `PATH` it auto-installs into `.venv/bin/` (or run `usersim setup-ngc` ahead of time, per the README).
- An `assets/` directory with one folder per probe (`assets/<probe>/`). These ship with the repo.

**What this writes:**

- `output/trajectories/locale=X/probe_family=Y/part-N.parquet`: directory-partitioned dataset, one row per simulated trajectory, with `simulation_outcome`, `simulation_traces`, `persona_uuid`, `trajectory_id`, `probe_family`, etc. as side-effect columns. Each locale's partition lands atomically immediately after that locale's batch completes: a crash mid-run leaves earlier locales' partitions intact and resumable.

**Available probes** (selectable via `PROBE_MIX` below; each writes the same trajectory schema, only the per-probe side channels differ):

| Family | Probe | Curated asset | When to use |
|---|---|---|---|
| General | `tool_calling` | `assets/tool_calling/toolsets_seed.parquet` | Tests native tool-call selection + arg synthesis on persona-grounded requests |
| General | `general_open_ended` | `assets/general_open_ended/topics/<locale>/topics.json` | Tests multi-turn conversational quality on open-ended topics |
| General | `general_educational` | `assets/general_educational/subjects/<locale>/subjects.json` | Tests pedagogical responses (level-calibration, scaffolded explanations) |
| Sovereign-AI | `sov_ai_facts` | `assets/sov_ai_facts/<locale>/` | Per-locale fact bank with curated ground truth + false-premise probes |
| Sovereign-AI | `sov_ai_dynamic` | `assets/sov_ai_dynamic/<locale>/` | Persona-driven exploration with no ground truth: surfaces hallucination / self-inconsistency |
| Sovereign-AI | `sov_ai_multilingual_parity` | `assets/sov_ai_multilingual_parity/sample.yaml` | Cross-locale matched-pair queries: surfaces respect-parity gaps across demographic strata |
| Safety | `safety_chat_pressure` | `assets/safety_chat_pressure/sample.yaml` | Multi-turn pressure loop with deterministic reframing rotation against curated refusal targets |
| Safety | `safety_agentic` | `assets/safety_agentic/sample.yaml` | Mock-environment tool-call probing for boundary violations / consequence disclosure / injection resistance |
| Financial-services | `financial_services` | `assets/financial_services/<locale>/` | Multi-institution support/advisory in two tiers (verifiable + dynamic): **opt-in + asset-backed**; a reference en_US bank ships today (see the "Financial-services probe" note below the config cell) |
| Identity | `identity_disclosure` | `assets/identity_disclosure/spec.yaml` | Asks the model under test who it is and who made it, in the row's language; each row records the answers that model may give |

The sovereign-AI, safety, `financial_services` and `identity_disclosure` probes ship with **AI-authored beta content** flagged `placeholder: true` on every entry: substantive locale-specific institutional facts, false-premise traps, attack-pattern targets, native-language taxonomies, and identity questions. The flag remains `true` until a native reviewer / SME audit attaches real citations. Downstream production-readiness scorecards refuse "ready" headlines for any cell that touched a placeholder-flagged entry: treat any sovereign-AI / safety claims out of this repo as beta signal until the gate flips.

## 🔧 Setup

In [ ]:
%load_ext autoreload
%autoreload 2

# --- Standard library ---
import getpass
import json
import logging
import os
import subprocess
import warnings
from collections import Counter
from pathlib import Path

# --- Third-party ---
import pandas as pd
from rich.console import Console

# --- usersim CLI helpers ---
from usersim.cli._models import (
    REQUIRED_SIMULATOR_ALIASES,
    apply_model_overrides,
    bundled_models_path,
    default_models_path,
    describe_models_source,
    load_models_config,
    print_resolved_models,
    prompt_for_provider_keys,
    require_aliases,
    smoke_test_models_or_raise,
)
from usersim.cli._env import describe_loaded_env, load_local_env
from usersim.cli._ngc import ensure_ngc_cli, ensure_ngc_org, ngc_available
from usersim.cli._pipeline import build_simulator_config_builder, known_probes

# --- Engine (asset + storage + preview helpers) ---
from usersim.engine.core._assets import packaged_assets_dir, probe_assets_dir
from usersim.engine.core.preview import (
    filter_preview_locale,
    preview_row,
    preview_summary,
)
from usersim.engine.core.storage import (
    existing_trajectory_ids,
    latest_run_id,
    list_partition_keys,
    new_run_id,
    read_partitioned_dataset,
    resolve_run_or_raise,
    run_subroot,
    write_locale_partition,
)

warnings.filterwarnings("ignore", message="urllib3.*chardet.*charset_normalizer")
logging.getLogger("usersim.engine").setLevel(logging.INFO)

# Credentials from .env.local / .env, so the kernel does not depend on what
# was exported in the shell that launched Jupyter. Anything already set wins,
# and any key still missing is prompted for by the Credentials cell below.
print("env:", describe_loaded_env(load_local_env()))

# --- Where the data lives ---
# Trajectories land as a directory-partitioned dataset rooted at
# TRAJECTORY_PATH (Hive-style: locale=X/probe_family=Y/part-N.parquet).
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# Banks ship inside the installed package, so resolve them from there rather
# than from the checkout. This works the same from a wheel install.
ASSETS_DIR = packaged_assets_dir()
OUTPUT_DIR = REPO_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRAJECTORY_PATH = OUTPUT_DIR / "trajectories"
TRAJECTORY_PATH.mkdir(parents=True, exist_ok=True)
TOOLSET_SEED_PATH = (
    probe_assets_dir("tool_calling", assets_root=ASSETS_DIR)
    / "toolsets_seed.parquet"
)

## ⚙️ Configuration

All knobs in one place. Edit and re-run from here.

In [ ]:
# --- Run knobs ------------------------------------------------------------
# Five locales spanning four scripts (Latin, Devanagari, Japanese, Hangul) and
# four regions, which is the point of the sovereign_ai preset below. The other
# shipped locales are en_IN, en_SG, fr_FR and hi_Latn_IN.
TARGET_LOCALES = ["en_US", "pt_BR", "hi_Deva_IN", "ja_JP", "ko_KR"]

# --- India multi-language stop-gap (opt-in) ------------------------------
# Extra Indian-language conversation locales that reuse the en_IN persona
# dataset + the en_IN asset banks. Conversations run in the target language
# (native script) or its romanized Latin twin; the verbatim turn-1 of
# asset-driven probes is machine-translated from en_IN and flagged
# USED_MACHINE_TRANSLATION (preview-only) until native banks are authored.
# These sample en_IN personas: no separate persona parquet is downloaded.
# Two lists so you can pick either or both (native scripts, romanized Latin,
# or the union). NOTE: the code after the language is the *script*, not the
# language: Devanagari ("Deva") is shared by Hindi, Marathi (mr) and Nepali
# (ne); Urdu ("Arab") uses the Arabic script. Append to TARGET_LOCALES, e.g.:
#   TARGET_LOCALES = TARGET_LOCALES + INDIA_NATIVE_LOCALES
#   TARGET_LOCALES = TARGET_LOCALES + INDIA_LATIN_LOCALES
#   TARGET_LOCALES = TARGET_LOCALES + INDIA_NATIVE_LOCALES + INDIA_LATIN_LOCALES
INDIA_NATIVE_LOCALES = [
    "bn_Beng_IN",  # Bengali
    "or_Orya_IN",  # Odia
    "ta_Taml_IN",  # Tamil
    "te_Telu_IN",  # Telugu
    "ur_Arab_IN",  # Urdu (Arabic script)
    "ml_Mlym_IN",  # Malayalam
    "mr_Deva_IN",  # Marathi (Devanagari)
    "gu_Gujr_IN",  # Gujarati
    "ne_Deva_IN",  # Nepali (Devanagari)
    "kn_Knda_IN",  # Kannada
    "pa_Guru_IN",  # Punjabi (Gurmukhi)
]
INDIA_LATIN_LOCALES = [
    "bn_Latn_IN", "or_Latn_IN", "ta_Latn_IN", "te_Latn_IN",
    "ur_Latn_IN", "ml_Latn_IN", "mr_Latn_IN", "gu_Latn_IN",
    "ne_Latn_IN", "kn_Latn_IN", "pa_Latn_IN",
]

LOCALES = TARGET_LOCALES            # append INDIA_NATIVE_LOCALES and/or INDIA_LATIN_LOCALES to widen
NUM_ROWS_PER_LOCALE = 5             # every row is a live multi-turn conversation, so raise this
                                    # deliberately: cost is len(LOCALES) x NUM_ROWS_PER_LOCALE
# Turns dominate run time: every turn is a user call, an assistant call and a
# judge call, so cost is roughly LOCALES x ROWS x MAX_TURNS. 5 matches the
# engine default. Raise it for financial_services, whose tasks need enough
# turns to search the KB, call a tool and act on the result; 8 to 10 is a
# better fit there. It is a single global setting, not per-probe, so a mixed
# run has to pick one number.
MAX_TURNS = 5
# Assistant-turn resampling budget: 1 = flag-only (evaluation default);
# >1 regenerates judge-rejected assistant turns (SDG runs).
MAX_ASSISTANT_ATTEMPTS = 1
RANDOM_SEED = 27                    # reproducible tool sampling
SIM_RUN_ID: str | None = None       # None: honor RESUME below.
                                    # Pinned (e.g. "1785990612"): write into THAT run, whether or
                                    # not it is the latest. This is how you finish a run you
                                    # stopped part-way -- RESUME can only reach the MOST RECENT
                                    # run, so once a newer one exists the older partial is
                                    # otherwise unreachable. Set LOCALES to the locales that are
                                    # still missing: the run-id cell prints which ones the run
                                    # already has. (Generation happens BEFORE the trajectory-id
                                    # skip, so leaving a finished locale in LOCALES re-pays its
                                    # full LLM cost and then discards the rows.)
RESUME = False                      # True:  resume into the most-recent existing run under TRAJECTORY_PATH;
                                    #        skip trajectory_ids already in that run and append new partitions.
                                    # False: mint a NEW run under TRAJECTORY_PATH/run=<unix-epoch>/ and write
                                    #        fresh partitions there. Old runs stay on disk untouched but are
                                    #        ignored by the default reader (which picks the latest run).

# --- Probe mix -----------------------------------------------------------
# Pick ONE preset by name. The presets cover the common runs; for
# anything else, set PROBE_MIX directly below (any subset of the
# registered probes; weights renormalize to 1.0). See the asset
# table at the top of this notebook for what each probe consumes.

# `sovereign_ai` is the default because its assets exist for all nine shipped
# locales, so the multi-locale default above runs as-is. `financial_services`
# has banks for en_US, en_IN and hi_Deva_IN only, and a run naming a locale
# without one stops at the pre-flight check rather than part-way through.
PROBE_PRESET = "sovereign_ai"   # one of: see PROBE_PRESETS below

# financial_services only (ignored by every other probe): probability
# [0,1] that a financial_services trajectory runs the DYNAMIC tier
# (generated open-ended question, reference-grounded judge) instead of
# the default VERIFIABLE tier (param-locked task, deterministic verifier).
# 0.0 = verifiable-only; 1.0 = dynamic-only; 0.3 = ~30% dynamic.
FINANCE_TIER_MIX = 0.5

# financial_services only: kb_search retrieval backend.
#   "hybrid": (default) normalised cosine over per-document embeddings blended
#              with lexical overlap. BOTH tiers retrieve from the KB
#              (tau-style). No locale ships embedding vectors, so this
#              retrieves lexically until you build them with gen-assets.
#   "dense" : cosine alone. An ABLATION, and a known-weak one on this corpus.
#   "golden": reasoning-isolation ABLATION that injects the verifiable task's gold
#              docs so retrieval quality is removed as a variable. The DYNAMIC
#              tier has no gold docs, so golden degrades to lexical there.
# Retrieval quality itself is reported separately in eval as the
# `financial_retrieval_recall` capability (gold-doc recall), so you can tell
# retrieval failures apart from reasoning failures without re-running.
FINANCE_RETRIEVAL_MODE = "hybrid"

PROBE_PRESETS: dict[str, dict[str, float]] = {
    "rlhf": {
        # "general_educational": 1.0#1/4,
        "general_open_ended": 1.0#1/4,
        # "sov_ai_dynamic": 1.0 #1/4,
        # "tool_calling": 1.0#1/4,
    },

    # tool_calling / general_open_ended / general_educational: the
    # general-purpose conversational probes (theme-driven; the
    # user-agent synthesises the user query from persona + theme rather
    # than injecting a curated asset). Feeds the capability dashboard's
    # general capability rows + the embedded Sim Health panel.
    "general": {
        "tool_calling": 1/3,
        "general_open_ended": 1/3,
        "general_educational": 1/3,
    },
    # Curated fact bank + population-probing taxonomy + cross-locale
    # query bank. Feeds the sovereign-AI capability rows in the
    # capability dashboard.
    "sovereign_ai": {
        "sov_ai_facts": 1/3,
        "sov_ai_dynamic": 1/3,
        "sov_ai_multilingual_parity": 1/3,
    },
    # Both safety probes. Feeds the safety_pressure +
    # agentic_boundary capability rows in the capability dashboard.
    "safety": {
        "safety_chat_pressure": 0.5,
        "safety_agentic": 0.5,
    },
    # Population-grounded, multi-institution financial support/advisory.
    # OPT-IN + asset-backed: needs a generated bank under
    # assets/financial_services/<locale>/ (a reference en_US bank ships
    # with the repo: keep LOCALES = ["en_US"], or generate other locales
    # first; see the "Financial-services probe" note below). Pair with
    # FINANCE_TIER_MIX (above) to dial the verifiable/dynamic tier split.
    # Feeds the financial_task_success* + dynamic_grounding capability rows.
    "financial_services": {
        "financial_services": 1.0,
    },
    # Asks the model under test who it is and who made it, and records the
    # answers that model may give (assets/identity_disclosure/spec.yaml).
    # The probe infers who made the assistant model from the model family name
    # in its id. For an id it cannot infer (a deployment name, a local path),
    # declare the model in a spec layer and point the override at it (see
    # assets/identity_disclosure/SCHEMA.md):
    #   os.environ["USERSIM_IDENTITY_DISCLOSURE_SPEC"] = str(REPO_ROOT / "my_models.local.yaml")
    "identity": {
        "identity_disclosure": 1.0,
    },
    # All eight probes. Surfaces every capability row in the
    # dashboard. Bump NUM_ROWS_PER_LOCALE for meaningful per-probe
    # sample sizes: the mix is uniform so each probe gets ~1/8 of
    # the rows.
    "all": {
        "tool_calling": 1/8,
        "general_open_ended": 1/8,
        "general_educational": 1/8,
        "sov_ai_facts": 1/8,
        "sov_ai_dynamic": 1/8,
        "sov_ai_multilingual_parity": 1/8,
        "safety_chat_pressure": 1/8,
        "safety_agentic": 1/8,
    },
    # Single-probe debug: useful when iterating on a specific probe /
    # scorer / bundle. Edit the key to debug another probe.
    "debug": {"safety_agentic": 1.0},
}

if PROBE_PRESET not in PROBE_PRESETS:
    raise KeyError(
        f"PROBE_PRESET={PROBE_PRESET!r} not in "
        f"{sorted(PROBE_PRESETS)}"
    )
PROBE_MIX = PROBE_PRESETS[PROBE_PRESET]
print(f"Selected PROBE_PRESET={PROBE_PRESET!r} → {sorted(PROBE_MIX)}")
print(
    f"Locales: {LOCALES} "
    f"({len(LOCALES)} × {NUM_ROWS_PER_LOCALE} rows "
    f"= {len(LOCALES) * NUM_ROWS_PER_LOCALE} trajectories)"
)
print(f"Available presets: {sorted(PROBE_PRESETS)}")
print(f"Available probes: {list(known_probes())}")

# --- Models TOML ---
# Switch providers by changing one word. Each shipped config runs every alias
# on one provider and needs only that provider's key:
#
#   "auto"       whatever `usersim` would use: $USERSIM_MODELS_CONFIG, then a
#                models.local.toml at or above the working directory, then
#                the bundled default
#   "nvidia"     build.nvidia.com            NVIDIA_API_KEY
#   "openai"     api.openai.com              OPENAI_API_KEY
#   "openrouter" openrouter.ai               OPENROUTER_API_KEY
#
# Per-alias model choices happen in the next cell; sampling parameters come
# from cli/model_catalog.py.
MODELS_PROFILE = "auto"

_SHIPPED = bundled_models_path().parent
MODELS_CONFIG_PATH = {
    "auto": default_models_path(),
    "nvidia": bundled_models_path(),
    "openai": _SHIPPED / "models_openai.toml",
    "openrouter": _SHIPPED / "models_openrouter.toml",
}[MODELS_PROFILE]
print("models config:", describe_models_source(MODELS_CONFIG_PATH))

### 🏦 Financial-services probe (opt-in, asset-backed)

`financial_services` is **not** in the default mix. It simulates census-grounded
users talking to a financial institution's support/advisory agent, grounded in
that institution's own knowledge base, and runs in two tiers:

- **verifiable**: param-locked tasks scored by a deterministic verifier.
- **dynamic**: generated open-ended questions scored by a reference-grounded judge.

To run it, select the preset and (optionally) dial the tier split via the two
knobs in the configuration cell above:

```python
PROBE_PRESET = "financial_services"
FINANCE_TIER_MIX = 0.3            # ~30% dynamic tier, the rest verifiable (0.0 = verifiable-only)
FINANCE_RETRIEVAL_MODE = "hybrid"  # default: dense+lexical KB retrieval for both tiers
```

The assistant grounds its answers by calling a `kb_search` tool that retrieves
whole documents from the institution's knowledge base (scoped to that one
institution). The default `dense` backend ranks the institution's precomputed
doc embeddings against the query vector, so **both** tiers retrieve from the KB
like a production support agent (it needs the `embedding_model` endpoint, and
degrades to lexical overlap if that's unreachable). `golden` is a
reasoning-isolation **ablation** that injects the verifiable task's gold
documents to remove retrieval quality as a variable: the dynamic tier has no
gold docs, so `golden` degrades to lexical there.

Retrieval quality is measured separately from task success: eval surfaces a
`financial_retrieval_recall` capability (fraction of gold docs the agent
actually retrieved) next to `financial_task_success`. Reading the two together
attributes a failure to retrieval (low recall) vs. policy/reasoning (high
recall but wrong tools/state), so you usually don't need the `golden` ablation
run to disentangle the two.

It is **asset-backed**: each locale needs a generated bank under
`assets/financial_services/<locale>/`. A reference `en_US` bank ships with the
repo, so keep `LOCALES = ["en_US"]` until you generate more. The banks are
produced offline (see [`asset_gen/notebooks/finance_assets.ipynb`](../asset_gen/notebooks/finance_assets.ipynb))
and managed with three CLI commands:

```bash
usersim gen-assets      --domain financial_services --locale en_US   # generate a bank
usersim validate-assets --domain financial_services --locale en_US   # offline structural checks
usersim evaluate-assets --domain financial_services --locale en_US   # DD-judge quality scorecard
```

The equivalent CLI simulate invocation (instead of this notebook) is:

```bash
usersim simulate --locale en_US --probe-mix financial_services=1.0 --finance-tier-mix 0.3
```

### Models: per-alias overrides

In [ ]:
# --- Per-alias model overrides ---
# Empty means use the models TOML as shipped. Add an entry to swap the model
# behind one alias, most often `assistant_model`, the model under test.
#
# Per-model fields (temperature / top_p / max_tokens / extra_body) re-resolve
# from cli/model_catalog.py on swap; per-alias fields (max_parallel_requests,
# timeout) carry over from the TOML row. A model the catalog does not know,
# with no provider able to serve it, raises ConfigError rather than guessing
# an endpoint, so add new ids to the catalog first.
#
# The MODELS load, key prompts and smoke check live in the Credentials cell
# below; this cell is only the override knob.

# Uncomment and edit to override. Values are a model id, or a dict carrying
# per-model fields alongside it:
MODEL_OVERRIDES: dict[str, str | dict] = {
    # "assistant_model": "nvidia/nemotron-3-super-120b-a12b",   # model under test
    # "user_model":      "openai/gpt-oss-20b",                  # simulated user
    # "judge_model":     "nvidia/nemotron-3.5-lightning-30b-a3b",
    # "assistant_model": {
    #     "model": "google/gemma-4-31b-it",
    #     "extra_body": {},   # drop the catalog's enable_thinking
    # },
}


## 🔑 Credentials

API keys for the model providers + NGC (for persona dataset downloads). The first run prompts via `getpass` (no echo); subsequent runs see the keys in `os.environ` and pass through. Provider keys (e.g. `NVIDIA_INFERENCE_HUB_KEY`) are prompted only when your `MODEL_OVERRIDES` route to a provider whose key isn't already set.

In [ ]:
# 1) Build-time NVIDIA API key (build.nvidia.com).
if not os.environ.get("NVIDIA_API_KEY"):
    key = getpass.getpass("Enter your NVIDIA API key (build.nvidia.com): ")
    if key:
        os.environ["NVIDIA_API_KEY"] = key
print("NVIDIA_API_KEY configured:", bool(os.environ.get("NVIDIA_API_KEY")))

# 2) NGC auth is a SEPARATE credential from the inference key. NGC CLI
#    4.x reads `NGC_CLI_API_KEY`; generate one at
#    https://org.ngc.nvidia.com/account/api-keys and export it before
#    running the persona download. NVIDIA_API_KEY will NOT satisfy it.
if not os.environ.get("NGC_CLI_API_KEY"):
    print("NGC_CLI_API_KEY not set - the persona download step will fail.")

# 3) Load models config + apply per-alias overrides so we know which
#    providers we'll need credentials for.
MODELS = load_models_config(MODELS_CONFIG_PATH)
require_aliases(MODELS, required=REQUIRED_SIMULATOR_ALIASES, context="01_simulate")
if MODEL_OVERRIDES:
    MODELS = apply_model_overrides(MODELS, MODEL_OVERRIDES)

# 4) Prompt for any provider env var that's needed by a CURRENTLY-USED
#    alias and not already set (e.g. NVIDIA_INFERENCE_HUB_KEY when hub
#    aliases are loaded). This is what was previously silently blocking
#    the cell -- now it's in its own clearly-labeled cell.
prompt_for_provider_keys(MODELS)

# 5) Resolved state (post-override).
print_resolved_models(MODELS)

# 6) Smoke-test each (model, provider) before burning LLM budget on the
#    simulator's substrate. Skip with SKIP_MODEL_SMOKE_TEST=True if the
#    providers rate-limit aggressively (~256 output tokens per alias).
#    timeout_sec is bumped above the 15s default because reasoning models
#    (gemma-4 enable_thinking, gpt-oss reasoning_effort=high, etc.) emit a
#    thinking trace before the answer and can exceed 15s even for the tiny
#    smoke prompt. This only gates connectivity; the actual run uses each
#    alias's own (much larger) timeout.
SKIP_MODEL_SMOKE_TEST = False
if not SKIP_MODEL_SMOKE_TEST:
    smoke_test_models_or_raise(MODELS, timeout_sec=30)

In [ ]:
# Make sure the locales we want are present locally; download anything missing.
# `TARGET_LOCALES` is defined in the Configuration cell above.
from usersim.engine.core.locale import persona_dataset_locale

personas_dir = Path.home() / ".data-designer" / "managed-assets" / "datasets"
personas_dir.mkdir(parents=True, exist_ok=True)

existing = {p.stem.lower() for p in personas_dir.glob("*.parquet")}
# India language-variants (e.g. ta_Taml_IN) have no persona parquet of their
# own (they sample en_IN personas), so resolve each target to the dataset it
# actually needs (prefer a native parquet if one has been dropped in, else the
# en_IN base) and dedupe. This never tries to download a non-existent
# `ta_Taml_IN` from NGC.
needed_datasets = sorted({
    persona_dataset_locale(loc, datasets_dir=personas_dir) for loc in TARGET_LOCALES
})
missing = [ds for ds in needed_datasets if ds.lower() not in existing]

if missing:
    # `data-designer download personas` shells out to the `ngc` binary;
    # on a fresh cluster node (a) it isn't installed, and (b) NGC CLI 4.x
    # refuses authenticated commands without an org context (even for
    # public-catalog resources, with `no-org` explicitly rejected when an
    # API key is set). The README's setup step (`usersim setup-ngc`)
    # handles the install; the org context is auto-discovered here so
    # the notebook is self-healing on cluster nodes that bypass setup.
    if not ngc_available():
        print("NGC CLI not found: installing into .venv/bin/ (one-time)…")
        ensure_ngc_cli()
    if not ensure_ngc_org():
        raise RuntimeError(
            "Could not auto-discover your NGC org. Set NGC_CLI_ORG to one "
            "of the orgs your API key is a member of (run `ngc org list` "
            "in a shell to see them), or run `ngc config set` once on this "
            "host to configure persistently."
        )
    print(f"Downloading missing persona datasets: {missing}")
    for loc in missing:
        subprocess.run(
            ["data-designer", "download", "personas", "--locale", loc],
            input="Y\n", text=True, check=True,
        )
print(f"Personas dataset ready at {personas_dir}")

## 🔁 Resumability check + run-id resolution

Every fresh simulation lands under its own `run=<unix-epoch>/` partition under `TRAJECTORY_PATH`. The reader defaults to "latest run" so downstream evaluation + reporting see the most recent results without configuration. Old runs stay on disk untouched, available for run-over-run comparison or as historical evidence.

Behaviour depends on `RESUME` (set in the configuration cell above):

- **`RESUME = True`**: resume into the most-recent existing run. Skip trajectory_ids already present in that run and append new partitions to it. Useful when iterating on a probe-mix expansion or recovering from a crashed run.
- **`RESUME = False`** (default): mint a new run id (`int(time.time())`) and start fresh under `TRAJECTORY_PATH/run=<that-id>/`. Old runs are not deleted (they keep being readable via `read_partitioned_dataset(TRAJECTORY_PATH, run="<id>")` or `run="all"`) but the default-latest read won't surface them.

The simulator's identity scheme (`trajectory_id` is content-hashed from persona + probe + seed + model identities + prompt version) means resumption is byte-stable: re-running the same configuration against the same panel produces the same `trajectory_id`s and the resume path is a no-op cost.

In [ ]:
if SIM_RUN_ID is not None:
    # Pinned run: finish a run you stopped part-way. RESUME reaches only the
    # LATEST run, so once a newer one exists the older partial is otherwise
    # unreachable -- this is the only way back to it.
    RUN_ID = resolve_run_or_raise(TRAJECTORY_PATH, SIM_RUN_ID, label="SIM_RUN_ID")
    already_done = existing_trajectory_ids(TRAJECTORY_PATH, run=RUN_ID)
    on_disk = sorted(list_partition_keys(run_subroot(TRAJECTORY_PATH, RUN_ID)))
    todo = [loc for loc in LOCALES if loc not in on_disk]
    print(
        f"SIM_RUN_ID={RUN_ID}: writing into that existing run; "
        f"{len(already_done)} trajectory_id(s) already present, will skip those."
    )
    print(f"  locales already on disk ({len(on_disk)}): {on_disk}")
    if todo:
        print(f"  LOCALES still missing from this run ({len(todo)}): {todo}")
    already_there = [loc for loc in LOCALES if loc in on_disk]
    if already_there:
        # Loud, because the skip happens AFTER generation: these locales would be
        # simulated in full and then thrown away.
        print(
            f"  WARNING: LOCALES still contains {len(already_there)} locale(s) this run "
            f"already has {already_there} -- they will be re-simulated at full LLM cost "
            f"and then discarded by the trajectory-id skip. Narrow LOCALES to the "
            f"missing ones above."
        )
elif RESUME:
    RUN_ID = latest_run_id(TRAJECTORY_PATH) or new_run_id()
    already_done = existing_trajectory_ids(TRAJECTORY_PATH, run=RUN_ID)
    print(
        f"RESUME=True: resuming into run={RUN_ID}; "
        f"{len(already_done)} trajectory_id(s) already present, will skip those."
    )
else:
    RUN_ID = new_run_id()
    already_done = set()
    print(
        f"RESUME=False: minting new run={RUN_ID} under {TRAJECTORY_PATH}/."
    )
    existing = latest_run_id(TRAJECTORY_PATH)
    if existing is not None and existing != RUN_ID:
        print(
            f"  (Previous run on disk: {existing}. Untouched. Read it back via "
            f"read_partitioned_dataset(TRAJECTORY_PATH, run='{existing}').)"
        )

## ▶️ Simulate

Loop over locales, generate trajectories, write each locale's batch atomically to its own partition under `output/trajectories/`. The simulator emits one row per trajectory; the row carries `simulation_outcome` (status / failure_class / counters / per-model resource), `simulation_traces` (per-turn telemetry), `persona_uuid`, and `trajectory_id`.

Per-locale atomic writes mean a crash mid-run (e.g., on the third locale) leaves the first two locales' partitions intact and resumable on the next invocation.

In [ ]:
from usersim.engine.config import ConversationSimulatorConfig
from usersim.engine.core.manifest import write_simulator_manifest
from usersim.cli._pipeline import DEFAULT_TOOL_CALLING_THEMES

total_written = 0
# The toolset seed is a tool_calling input. Passing it when the mix has no
# tool_calling probe is rejected as a contradictory override, so only send it
# when that probe is actually selected.
_seed_path = (
    TOOLSET_SEED_PATH
    if "tool_calling" in PROBE_MIX and TOOLSET_SEED_PATH.exists()
    else None
)

for locale in LOCALES:
    print(f"\nSimulating {NUM_ROWS_PER_LOCALE} rows for {locale}...")
    data_designer, builder = build_simulator_config_builder(
        locale=locale,
        models=MODELS,
        assets_dir=ASSETS_DIR,
        max_turns=MAX_TURNS,
        max_assistant_attempts=MAX_ASSISTANT_ATTEMPTS,
        probe_mix=PROBE_MIX,
        finance_tier_mix=FINANCE_TIER_MIX,
        finance_retrieval_mode=FINANCE_RETRIEVAL_MODE,
        toolset_seed_path=_seed_path,
        random_seed=RANDOM_SEED,
    )
    result = data_designer.create(builder, num_records=NUM_ROWS_PER_LOCALE)
    df = result.load_dataset()
    if already_done:
        df = df[~df["trajectory_id"].isin(already_done)]
    if df.empty:
        print(f"  (locale={locale}: no new trajectories, all already in dataset)")
        continue
    # Atomic per-locale write: this locale's partition lands on disk
    # before the next locale starts. A crash mid-run leaves earlier
    # locales' partitions intact and resumable.
    write_locale_partition(df, TRAJECTORY_PATH, locale=locale, run_id=RUN_ID)
    total_written += len(df)
    print(f"  wrote {len(df)} new trajectories for {locale}")

print(f"\nWrote {total_written} new trajectories to {TRAJECTORY_PATH}")

# Write the run manifest. ``RUN_ID`` is the unix-epoch start time
# (see ``new_run_id``), so it doubles as ``started_at_epoch`` and the
# manifest's runtime is finished_at - RUN_ID. First-writer-wins:
# resuming into a run that already has a manifest is a noop.
sim_cfg_for_manifest = ConversationSimulatorConfig(
    name="conversation_messages",
    locale="placeholder",
    max_turns=MAX_TURNS,
    max_assistant_attempts=MAX_ASSISTANT_ATTEMPTS,
    context_compression=True,
    compression_window=1,
    random_seed=RANDOM_SEED,
)
# Describe the RUN, not just this pass. On a resumed run LOCALES holds only the
# locales this pass filled in, so recording it verbatim would understate a run that
# already has others on disk -- and the manifest is first-writer-wins, so there is no
# second chance to correct it. Union with what is actually there.
_run_locales = sorted(
    set(LOCALES) | set(list_partition_keys(run_subroot(TRAJECTORY_PATH, RUN_ID)))
)
manifest_path = write_simulator_manifest(
    run_id=RUN_ID,
    started_at_epoch=int(RUN_ID),
    trajectory_root=TRAJECTORY_PATH,
    models_config=MODELS,
    models_config_path=MODELS_CONFIG_PATH,
    sim_config=sim_cfg_for_manifest,
    locales_requested=_run_locales,
    probe_mix=PROBE_MIX,
    tool_calling_themes=DEFAULT_TOOL_CALLING_THEMES,
    toolset_seed_path=TOOLSET_SEED_PATH if TOOLSET_SEED_PATH.exists() else None,
    assets_dir=ASSETS_DIR,
    trajectories_requested={l: NUM_ROWS_PER_LOCALE for l in _run_locales},
    source_kind="notebook",
    source_invocation="notebooks/01_simulate.ipynb",
)
print(f"Wrote run manifest to {manifest_path}")

## ✅ Quick sanity check

Confirm the output dataset has the expected shape: every row carries a unique `trajectory_id`, a structured `simulation_outcome`, and the `probe_family` / `locale` columns the reporting layer reads.

In [ ]:
df = read_partitioned_dataset(TRAJECTORY_PATH)


def _outcome(v):
    """Decode the simulation_outcome JSON cell to a dict (or {} on miss)."""
    if isinstance(v, str):
        try:
            return json.loads(v)
        except (json.JSONDecodeError, TypeError):
            return {}
    return v if isinstance(v, dict) else {}


outcomes = df["simulation_outcome"].map(_outcome)

# --- Coverage --------------------------------------------------------
print(f"Total rows: {len(df)}")
print(f"Unique trajectory_ids: {df['trajectory_id'].nunique()}")
print(f"Probe families:       {df['probe_family'].value_counts().to_dict()}")
print(f"Locales:              {df['locale'].value_counts().to_dict()}")

# --- Status distribution ---------------------------------------------
status_counts = outcomes.map(lambda o: o.get("status")).value_counts(dropna=False).to_dict()
print(f"\nStatus distribution:  {status_counts}")

# --- Per-(probe_family × status) cross-tab ---------------------------
# Quick read of "which probes are healthy?": a table of trajectory
# counts indexed by probe_family with status as columns. Useful for
# the `all` preset where 8 probes mix in the same dataset.
xtab = (
    df.assign(status=outcomes.map(lambda o: o.get("status") or "<missing>"))
      .groupby(["probe_family", "status"]).size().unstack(fill_value=0)
)
print("\nPer-probe × status:\n")
print(xtab.to_string())

# --- Per-(probe_family × locale) cross-tab ---------------------------
# Locale coverage per probe: useful to spot probes that are
# under-represented in particular locales (e.g. shorter persona pools
# yielding fewer trajectories for hi_Deva_IN, or a probe that didn't
# fan out to every locale on this run).
locale_xtab = (
    df.groupby(["probe_family", "locale"]).size().unstack(fill_value=0)
)
locale_xtab["TOTAL"] = locale_xtab.sum(axis=1)
locale_xtab.loc["TOTAL"] = locale_xtab.sum(axis=0)
print("\nPer-probe × locale:\n")
print(locale_xtab.to_string())

# --- Per-(locale × status) cross-tab ---------------------------------
# Companion view to probe × status: surfaces locales that produce
# disproportionately many failures or warnings (often a sign of
# locale-specific judge / asset issues).
locale_status_xtab = (
    df.assign(status=outcomes.map(lambda o: o.get("status") or "<missing>"))
      .groupby(["locale", "status"]).size().unstack(fill_value=0)
)
print("\nPer-locale × status:\n")
print(locale_status_xtab.to_string())

# --- Failure attribution (only when there are failures) --------------
failed = [o for o in outcomes if isinstance(o, dict) and o.get("status") == "failed"]
if failed:
    print(f"\nFailure breakdown ({len(failed)} failed trajectory(ies)):")
    fail_keys = Counter(
        (o.get("failure_class"), o.get("failure_attribution"))
        for o in failed
    )
    for (cls, attr), n in sorted(fail_keys.items(), key=lambda kv: -kv[1]):
        print(f"  {n:>3d}  {cls or '?':<30s}  attributed to {attr or '?'}")

# --- Warning kinds (placeholder warnings + others) -------------------
warn_kinds = Counter()
for o in outcomes:
    if not isinstance(o, dict):
        continue
    for w in o.get("warnings") or []:
        if isinstance(w, dict):
            warn_kinds[w.get("kind", "<unknown>")] += 1
if warn_kinds:
    print(f"\nWarning kinds across all trajectories:")
    for kind, n in sorted(warn_kinds.items(), key=lambda kv: -kv[1]):
        print(f"  {n:>3d}  {kind}")
    if any(k.startswith("used_placeholder_") for k in warn_kinds):
        print(
            "  (placeholder warnings are expected on sovereign-AI / safety "
            "runs until reviewer-vetted asset banks land: downstream "
            "scorecards will refuse 'production-ready' headlines for "
            "tainted cells)"
        )

## 👀 Preview conversations

Inspect a handful of generated trajectories with the rich-formatted preview from `usersim.engine.core.preview`. The test protocol is called a **"probe"** (e.g. `tool_calling`, `safety_agentic`); the trajectory dataset's per-row columns are named accordingly: `probe_type`, `probe_family`, `probe_variant`.

Two views:

- **`preview_summary(df)`**: one-row-per-trajectory table with `# / Status / Probe / Style / Turns / Tools / Country / Language / Age / Sex / Persona`. `Country` (with flag) and `Language` are broken out from the combined `Locale` column for easier filtering; `Age` + `Sex` slot in just before `Persona` as the demographic distinguishers reviewers want at a glance. The status badge is tri-state (`PASS` green / `WARN` yellow / `FAIL` red). The footer line breaks down clean / with-warnings / failed counts so a placeholder-tainted row no longer hides under a green badge.
- **`preview_row(row)`**: full per-trajectory view, top-to-bottom:
  1. **Header rule** colour-matched to status, fields ordered to mirror the summary table (`# / STATUS / probe_name / interaction-disclosure / turns / tools / 🇨🇨 Country/Language / Persona / (grounded)`), followed by one `⚠ warning_kind` chip per simulation-quality `WarningKind` flag (e.g. `id_fabrication`, `poor_assistant_response×3`). The `#` is lifted from the row's pandas index so every per-row preview maps back to the summary table's `#` column. Variant + family + description live in the Probe panel below: keeping the rule short. Placeholder-asset warnings (`used_placeholder_query` / `used_placeholder_target` / `used_placeholder_agentic_action` / `used_placeholder_fact` / `used_placeholder_taxonomy`) are intentionally filtered: they're an asset-bank concern (the curated bank isn't reviewer-blessed yet), not a simulation-quality concern, and downstream reporting bundles already gate production-readiness on them via dedicated counters.
  2. **Side-channel detail line** for sovereign-AI / safety probes: `id=…  sub_protocol=…  attempted_actions=N`.
  3. **Probe panel** (always renders): what is being tested. Carries the probe name, family (`general` / `sov_ai` / `safety`), variant (when distinct from the probe name), and a short hardcoded description.
  4. **Persona panel**: who is being tested. Location / age / sex (when present) / education / occupation / interaction-disclosure styles, plus OCEAN traits with magnitude pills: `low` / `very_low` / `high` / `very_high` get coloured backgrounds (cool = below baseline, warm = above), `average` stays dim. Pills carry magnitude only: no semantic green/red, since "low neuroticism" is good and "high neuroticism" is concerning.
  5. **Probe Inputs panel** for sovereign-AI / safety probes, which curated asset drove the run. Resolved metadata (sanctioned/tempted action names, query concern axis, fact category, etc.) lifted from `conversation_metadata` so you can disambiguate `id=AR-UNSAN-001` without grepping the bank.
  6. **Full Conversation panel**: what happened. Turn-by-turn render with role emojis (`👤` user / `🤖` assistant / `📡` tool response / `🔧` inline tool calls). Each assistant turn carries an inline `✓` / `✗ rating` chip lifted from the in-sim `assistant_judge_ratings` so you can see where things degraded without scrolling.
  7. **Attempted Actions table** for `safety_agentic`: per-call audit with category × in-allowlist colour coding (`sanctioned` green / `tempted` yellow / `fabricated` red).
  8. **Trajectory metadata footer** (dim) with `trajectory_id` (for dataset grep), `probe_variant`, per-model call/token counts, `wall_clock_s`, bank versions, and code SHA.

The reader's mental flow is *what is being tested* (Probe panel) → *who is being tested* (Persona panel) → *which curated asset* (Probe Inputs panel) → *what happened* (Full Conversation panel). The renderer works for every probe family: every probe writes the same OpenAI-style `conversation_messages` list, so the conversation panel renders consistently regardless of which probe produced the row. Trajectory-judgment and eval-scores panels are skipped on the trajectory dataset: those live in the eval dataset from `02_evaluate_simulation.ipynb`.

In [ ]:
# How many trajectories to render in the per-row view. Bump up for
# longer reviews.
MAX_PREVIEW_ROWS = 10
# Cap the summary table to N rows per locale so a large run (e.g.
# 1k+ trajectories across 8 locales) stays scannable. Set to None to
# render every row.
SUMMARY_ROWS_PER_LOCALE = 10
MAX_MESSAGE_LENGTH = 600       # truncate long messages in the conversation panel
MAX_MESSAGES_PER_TRAJECTORY = 30  # cap displayed messages per row
# Restricts only the detailed per-record previews. The summary table
# is capped independently via SUMMARY_ROWS_PER_LOCALE. Accepts locale
# codes ("en_US", "ja_JP") or language labels ("English", "Japanese").
# Set to None for all rows.
PREVIEW_LOCALE = "English"

# console = Console(width=200, height=50)
console = Console(width=175)
if SUMMARY_ROWS_PER_LOCALE is None:
    _summary_df = df
else:
    _summary_df = df.groupby("locale", group_keys=False).apply(
        lambda g: g.sample(n=min(SUMMARY_ROWS_PER_LOCALE, len(g)), random_state=0)
    )
preview_summary(_summary_df, console)

# Shuffle before previewing so a single rerun surfaces a varied
# cross-section across probes / locales / personas instead of
# clustering on whatever probe the simulator finished last.
# Each row's pandas index (preserved through .sample) is rendered
# as "#N" in the per-row header rule so you can map every preview
# back to the summary table's "#" column.
preview_df = filter_preview_locale(df, PREVIEW_LOCALE)
preview_subset = preview_df.sample(n=min(MAX_PREVIEW_ROWS, len(preview_df)))
for _, row in preview_subset.iterrows():
    preview_row(
        row,
        console,
        max_message_length=MAX_MESSAGE_LENGTH,
        max_messages=MAX_MESSAGES_PER_TRAJECTORY,
    )

## 🎯 Next step

Open `02_evaluate_simulation.ipynb` to score the trajectories with the trajectory-evaluator and render the capability dashboard against this trajectory dataset. The dashboard surfaces:

- **Capability heatmap**: capability × locale cells (`assistant_quality`, `multilingual_reliability`, `tool_use`, `sov_ai_facts`, `sov_ai_dynamic`, `sov_ai_multilingual_parity`, `safety_pressure`, `agentic_boundary`, ...) with score / threshold / state (ready / blocked / inconclusive / missing) and click-through to the underlying conversations.
- **Triage queue**: failed cells sorted by impact × evidence strength, each with a suggested next action.
- **Sim Health panel** ("did the simulator do its job?") failure-mode taxonomy, D1–D4 + MATTR behavioral diagnostics, persona-grounding rate, resource profile. No model calls.
- **Coverage matrix**: locale × probe trajectory and evaluator-cell counts, plus scorer-skip / scorer-error counts so coverage gaps surface at a glance.
- **Actors panel**: manifest-derived alias → model identities (user / assistant / judge / api / summary) so reviewers can attribute results to the actual model under test.

Capability rows render as `not measured` until the evaluator dataset carries the scorer rows that feed them, so a tool-calling-only trajectory dataset will surface only the capabilities that probe family supports. Pick the probe mix above accordingly.